# AG de recolección de 15 mercados de Chiclayo

Ejecutar las celdas en orden. Compatible con Colab; la ejecución de entrega se hizo localmente, no en una sesión de Colab. Incluye los datos y el motor de esta versión, sin instalar paquetes. Los kg son estimados; Q=10000 kg, K=2, J=3 son un escenario académico. Se informa mejor solución encontrada, no óptimo global. Fuentes completas en docs/Fuentes_y_aportes.md del proyecto.


In [ ]:
from pathlib import Path
import json
import sys
RAIZ_NOTEBOOK = Path.cwd()
archivos = {'con_comentarios/sistema/lector_datos.py': '"""Lectura y validación de mercados, distancias y configuración del escenario."""\n\n# Solo se importan módulos que ya vienen con Python (biblioteca estándar).\nimport csv          # leer el dataset de mercados (.csv)\nimport hashlib      # huella SHA-256 del dataset para saber con qué datos se corrió\nimport json         # leer la matriz de distancias (.json)\nimport math         # comprobar que las distancias sean números finitos\nfrom decimal import Decimal  # convertir kg con decimales sin errores de redondeo\nfrom pathlib import Path     # manejar rutas de archivos en Windows/Linux\n\n\n# ============================================================\n# UBICACIÓN DE LOS DATOS\n# ============================================================\n\n# Carpeta raíz del proyecto: la primera carpeta superior que contiene datos/.\nRAIZ = next(p for p in Path(__file__).resolve().parents if (p / "datos").is_dir())\n\n\n# ============================================================\n# PASO 0. PREPARACIÓN DE DATOS\n# ============================================================\n\n# Representa los kg como enteros en centésimas para no perder masa al sumar y restar.\ndef centikg(valor):\n    """Representa 0,01 kg con un entero para conservar exactamente la masa."""\n    # Ejemplo: 4690.98 kg -> 469098 centésimas de kg. Trabajar con enteros evita\n    # que al sumar y restar decimales se pierdan o aparezcan gramos de basura.\n    d = Decimal(str(valor))\n\n    # Se rechazan valores negativos, cero, infinitos o con más de dos decimales.\n    if not d.is_finite() or d <= 0 or d * 100 != (d * 100).to_integral_value():\n        raise ValueError("La capacidad/demanda debe ser positiva y tener hasta dos decimales.")\n\n    return int(d * 100)\n\n\n# Lee los 15 mercados y la matriz vial, y comprueba que sean coherentes entre sí.\ndef cargar_datos(carpeta=None):\n    """Valida la entrada y conserva la dirección de cada distancia vial."""\n    # Si no se indica carpeta, se usa datos/ del proyecto.\n    carpeta = Path(carpeta) if carpeta else RAIZ / "datos"\n    csv_path = carpeta / "15_mercados_chiclayo_11_columnas.csv"\n\n    # Cada fila del CSV es un mercado: id, nombre, distrito, puestos, kg/día, coordenadas.\n    with csv_path.open(encoding="utf-8-sig", newline="") as f:\n        mercados = list(csv.DictReader(f))\n\n    # La matriz vial tiene 17 puntos: 0 = base EPSEL, 1..15 = mercados, 16 = botadero Reque.\n    vial = json.loads((carpeta / "matriz_vial.json").read_text(encoding="utf-8"))\n    ids = [m["id_mercado"] for m in mercados]\n\n    # Los IDs de mercado serán los genes, por eso no pueden repetirse.\n    if not ids or len(ids) != len(set(ids)):\n        raise ValueError("El dataset debe contener IDs únicos.")\n\n    # El orden del CSV debe coincidir con las filas 1..15 de la matriz.\n    if vial["orden_ids"][1:-1] != ids:\n        raise ValueError("El orden del CSV no coincide con la matriz.")\n\n    n = len(ids) + 2  # 15 mercados + base + botadero = 17\n    matriz = vial["distancias_m"]\n\n    # La matriz debe ser cuadrada 17x17.\n    if len(matriz) != n or any(len(row) != n for row in matriz):\n        raise ValueError("La matriz debe ser cuadrada e incluir base y descarga.")\n\n    # matriz[i][j] = metros por carretera de i a j. Puede ser distinta de matriz[j][i]\n    # porque las calles tienen sentido de circulación (matriz dirigida).\n    for i, row in enumerate(matriz):\n        for j, value in enumerate(row):\n            if (\n                isinstance(value, bool)\n                or not isinstance(value, (int, float))\n                or not math.isfinite(value)\n                or value < 0\n                or (i != j and value == 0)\n            ):\n                raise ValueError("Hay una distancia vial faltante o inválida.")\n\n        # La distancia de un punto a sí mismo es cero.\n        if row[i] != 0:\n            raise ValueError("La diagonal de la matriz debe ser cero.")\n\n    # Las coordenadas del CSV deben ser las mismas con las que se calculó la matriz.\n    for m, p in zip(mercados, vial["puntos"][1:-1]):\n        if float(m["latitud_wgs84"]) != p["latitud"] or float(m["longitud_wgs84"]) != p["longitud"]:\n            raise ValueError("Las coordenadas no coinciden con la matriz vial.")\n\n    # Demanda de cada mercado (kg/día estimados) en centésimas de kg.\n    demandas = {m["id_mercado"]: centikg(m["residuos_kg_dia_estimados"]) for m in mercados}\n\n    # \'indices\' traduce cada ID de mercado a su fila/columna en la matriz (1..15).\n    return {\n        "mercados": mercados,\n        "ids": ids,\n        "demandas": demandas,\n        "matriz": matriz,\n        "indices": {i: k + 1 for k, i in enumerate(ids)},\n        "vial": vial,\n        "sha256_csv": hashlib.sha256(csv_path.read_bytes()).hexdigest(),\n    }\n\n\n# ============================================================\n# VALIDACIÓN DE LA CONFIGURACIÓN\n# ============================================================\n\n# Revisa los parámetros del escenario y del AG antes de empezar a evolucionar.\ndef validar_config(config, datos):\n    """Rechaza parámetros inválidos y flota con capacidad total insuficiente."""\n    c = dict(config)\n\n    # Hiperparámetros enteros y sus rangos permitidos.\n    for k, minimo, maximo in [\n        ("camiones", 1, 30),\n        ("viajes_por_camion", 1, 30),\n        ("poblacion", 4, 500),\n        ("generaciones", 1, 3000),\n        ("elite", 1, 50),\n        ("torneo", 2, 50),\n        ("paciencia", 1, 3000),\n        ("semilla", 0, 2**32 - 1),\n    ]:\n        v = c.get(k)\n        if isinstance(v, bool) or not isinstance(v, int) or not minimo <= v <= maximo:\n            raise ValueError(f"{k}: se requiere un entero entre {minimo} y {maximo}.")\n\n    # Las probabilidades de cruce y mutación deben estar entre 0 y 1.\n    for k in ("prob_cruce", "prob_mutacion"):\n        v = c.get(k)\n        if isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) or not 0 <= v <= 1:\n            raise ValueError(f"{k} debe estar entre 0 y 1.")\n\n    # No se pueden conservar más élites que individuos ni hacer torneos más grandes que la población.\n    if c["elite"] >= c["poblacion"] or c["torneo"] > c["poblacion"]:\n        raise ValueError("Elitismo menor que población y torneo no mayor que población.")\n\n    if isinstance(c.get("capacidad_kg"), bool):\n        raise ValueError("Capacidad inválida.")\n\n    q = centikg(c.get("capacidad_kg"))  # capacidad Q por viaje (10 000 kg -> 1 000 000)\n    if q > 10000000:\n        raise ValueError("Capacidad máxima: 100 000 kg.")\n\n    # Viajes mínimos necesarios = techo(demanda total / Q). Si la flota (camiones x viajes)\n    # no alcanza, el problema no tiene solución y se avisa antes de evolucionar.\n    total = sum(datos["demandas"].values())\n    necesarios = (total + q - 1) // q\n    if necesarios > c["camiones"] * c["viajes_por_camion"]:\n        raise ValueError(\n            f\'Capacidad insuficiente: se necesitan {necesarios} viajes y solo hay {c["camiones"] * c["viajes_por_camion"]}.\'\n        )\n\n    return c\n', 'con_comentarios/algoritmo_genetico/motor_genetico.py': '"""ALGORITMO GENÉTICO COMPLETO: empezar a leer arriba y continuar hacia abajo.\n\n1. La primera función, optimizar, muestra el ciclo con los PASOS 0 a 7.\n2. Debajo están los detalles de las cinco operaciones que ese ciclo utiliza.\n\nLa interfaz llama a optimizar al solicitar una optimización. Las funciones\nde apoyo pueden estar debajo porque Python carga el módulo antes de la llamada.\nTodo el AG está implementado aquí con Python estándar.\n"""\n\nimport random  # elecciones aleatorias reproducibles mediante una semilla\nimport time    # medición del tiempo de ejecución\n\nfrom sistema.lector_datos import centikg, validar_config\n\n\n# ============================================================\n# EMPIEZA AQUÍ: CICLO COMPLETO DEL ALGORITMO GENÉTICO\n# ============================================================\n\ndef optimizar(datos, config, progreso=None):\n    """Ejecuta población, evaluación, torneo, OX, swap y reemplazo con elitismo."""\n    # ----- PASO 0. PREPARAR Y VALIDAR LA CONFIGURACIÓN -----\n    inicio_t = time.perf_counter()\n    c = validar_config(config, datos)\n\n    # Generador aleatorio propio con semilla: la misma semilla produce el mismo resultado.\n    rng = random.Random(c["semilla"])\n\n    # ----- PASO 1. INICIALIZACIÓN DE LA POBLACIÓN -----\n    # Cada individuo (cromosoma) es una permutación de los 15 IDs de mercado.\n    # Gen = un mercado; el orden de los genes = el orden en que se atienden.\n    referencia = vecino_cercano(datos)\n    poblacion = [referencia]\n\n    # Una solución heurística y las restantes aleatorias forman la población inicial.\n    for _ in range(c["poblacion"] - 1):\n        individuo = datos["ids"][:]\n        rng.shuffle(individuo)  # orden aleatorio de los 15 mercados\n        poblacion.append(individuo)\n\n    # Memoria de costos ya calculados: si un orden se repite, no se vuelve a decodificar.\n    cache = {}\n\n    # ----- PASO 2. FUNCIÓN DE EVALUACIÓN -----\n    # Costo = metros totales del plan que produce el decodificador (menor es mejor).\n    def costo(individuo):\n        key = tuple(individuo)\n        if key not in cache:\n            cache[key] = decodificar(individuo, datos, c, detalle=False)\n        return cache[key]\n\n    # Mejor solución encontrada hasta el momento (empieza siendo la referencia).\n    mejor, mejor_costo, sin_mejora, historial = referencia[:], costo(referencia), 0, []\n    motivo = "limite_generaciones"\n\n    # CICLO DE GENERACIONES: repetir evaluación, reproducción y parada.\n    for generacion in range(c["generaciones"] + 1):\n\n        # ----- PASO 2. EVALUAR Y ORDENAR LA POBLACIÓN -----\n        # Minimizar distancia equivale a maximizar 1/(1 + distancia en km).\n        poblacion.sort(key=costo)  # el primero es el de menor distancia\n        actual = costo(poblacion[0])\n\n        # Si el mejor de esta generación supera al mejor histórico, se guarda.\n        if actual < mejor_costo - 1e-8:\n            mejor, mejor_costo, sin_mejora = poblacion[0][:], actual, 0\n        elif generacion > 0:\n            sin_mejora += 1\n\n        # Se registra la convergencia: mejor km y promedio de la población.\n        valores = [costo(p) / 1000 for p in poblacion]\n        historial.append(\n            {\n                "generacion": generacion,\n                "mejor_km": mejor_costo / 1000,\n                "promedio_km": sum(valores) / len(valores),\n                "fitness_mejor": 1 / (1 + mejor_costo / 1000),\n            }\n        )\n\n        if progreso:\n            progreso(historial[-1])  # avisa a la interfaz web para dibujar la curva\n\n        # Producir otra generación solo si aún queda presupuesto y no hay estancamiento.\n        continuar = sin_mejora < c["paciencia"] and generacion < c["generaciones"]\n        if continuar:\n            # Reservar dos plazas para las élites antes de producir descendientes.\n            nueva = [p[:] for p in poblacion[: c["elite"]]]  # los 2 mejores se copian sin cambios\n\n            while len(nueva) < c["poblacion"]:\n                # PASO 3: se eligen dos padres con dos torneos.\n                a = seleccionar_torneo(poblacion, costo, c["torneo"], rng)\n                b = seleccionar_torneo(poblacion, costo, c["torneo"], rng)\n\n                # PASO 4: con probabilidad prob_cruce (0,90) se cruzan con OX;\n                # si no, los hijos son copias de los padres.\n                hijos = (\n                    [cruce_ox(a, b, rng), cruce_ox(b, a, rng)]\n                    if rng.random() < c["prob_cruce"]\n                    else [a[:], b[:]]\n                )\n\n                for hijo in hijos:\n                    # PASO 5: cada hijo muta con probabilidad prob_mutacion (0,20).\n                    if rng.random() < c["prob_mutacion"]:\n                        hijo = mutar(hijo, rng)\n                    nueva.append(hijo)\n\n                    # Se detiene al completar el tamaño de la población.\n                    if len(nueva) == c["poblacion"]:\n                        break\n\n            # ----- PASO 6. REEMPLAZO CON ELITISMO -----\n            # La nueva generación reemplaza a la anterior.\n            poblacion = nueva\n\n        # ----- PASO 7. COMPROBAR SI TERMINA EL ALGORITMO -----\n        if sin_mejora >= c["paciencia"]:\n            motivo = "estancamiento"\n            break\n        if generacion == c["generaciones"]:\n            break\n\n    # ----- RESULTADO -----\n    # Se decodifican con detalle la referencia y la mejor solución para compararlas.\n    base = decodificar(referencia, datos, c)\n    plan = decodificar(mejor, datos, c)\n\n    return {\n        "configuracion": c,\n        "referencia": base,\n        "solucion": plan,\n        # Porcentaje de kilómetros ahorrados frente al vecino más cercano.\n        "mejora_porcentaje": 100 * (base["distancia_km"] - plan["distancia_km"]) / base["distancia_km"],\n        "historial": historial,\n        "generaciones_ejecutadas": len(historial) - 1,\n        "motivo_parada": motivo,\n        "evaluaciones_distintas": len(cache),\n        "tiempo_segundos": time.perf_counter() - inicio_t,\n        "sha256_dataset": datos["sha256_csv"],\n        "alcance": "Mejor solucion encontrada en la familia del decodificador; sin garantia de optimo global.",\n    }\n\n\n# ============================================================\n# DETALLE DE LAS OPERACIONES UTILIZADAS EN EL CICLO ANTERIOR\n# ============================================================\n\n# ============================================================\n# APOYO 1. REFERENCIA: VECINO MÁS CERCANO\n# ============================================================\n\n# Construye el orden "vecino más cercano": siempre ir al mercado pendiente más próximo.\ndef vecino_cercano(datos):\n    """Referencia reproducible: elige el mercado no visitado más cercano."""\n    # Partiendo de la base (punto 0), siempre se va al mercado pendiente más cercano.\n    # Este orden sirve de punto de comparación y entra como un individuo de la población inicial.\n    pendientes = set(datos["ids"])\n    actual, orden = 0, []\n\n    while pendientes:\n        # Se busca la menor distancia desde el punto actual; si hay empate, gana el ID menor.\n        siguiente = min(pendientes, key=lambda i: (datos["matriz"][actual][datos["indices"][i]], i))\n        orden.append(siguiente)\n        pendientes.remove(siguiente)\n        actual = datos["indices"][siguiente]\n\n    return orden\n\n\n# ============================================================\n# APOYO 2. CONSTRUCCIÓN DE VIAJES Y APTITUD\n# ============================================================\n\n# Convierte un cromosoma (orden de mercados) en viajes reales y calcula su distancia.\ndef decodificar(orden, datos, config, detalle=True):\n    """Llena cada viaje antes de iniciar el siguiente y conserva toda la demanda.\n\n    Cada viaje es base -> mercados -> descarga -> base. Si la demanda de un\n    mercado excede el espacio libre, se continúa ese mercado en el viaje siguiente.\n    Los camiones se asignan por turnos; no se optimizan horarios ni balance de carga.\n    """\n    # El cromosoma debe contener los 15 mercados exactamente una vez (una permutación).\n    if len(orden) != len(datos["ids"]) or set(orden) != set(datos["ids"]):\n        raise ValueError("El cromosoma debe ser una permutación de todos los mercados.")\n\n    q = centikg(config["capacidad_kg"])  # capacidad por viaje\n    matriz, indice = datos["matriz"], datos["indices"]\n    descarga = len(matriz) - 1  # último punto = botadero de Reque\n    viajes, paradas = [], []\n\n    # actual: punto donde está el camión (0 = base); carga: kg a bordo;\n    # distancia: metros del viaje en curso; total: metros de todo el plan; numero: viajes cerrados.\n    actual, carga, distancia, total, numero = 0, 0, 0.0, 0.0, 0\n\n    # Termina el viaje en curso: ir a descargar a Reque y regresar a la base.\n    def cerrar():\n        nonlocal actual, carga, distancia, total, numero, paradas\n        distancia += matriz[actual][descarga] + matriz[descarga][0]\n        total += distancia\n        numero += 1\n\n        # Restricción de flota: no se pueden hacer más viajes que camiones x viajes por camión.\n        if numero > config["camiones"] * config["viajes_por_camion"]:\n            raise ValueError("El plan supera el presupuesto de viajes.")\n\n        if detalle:\n            # Los viajes se reparten por turnos: viaje 1 -> camión 1, viaje 2 -> camión 2, etc.\n            viajes.append(\n                {\n                    "viaje": numero,\n                    "camion": (numero - 1) % config["camiones"] + 1,\n                    "turno_camion": (numero - 1) // config["camiones"] + 1,\n                    "carga_kg": carga / 100,\n                    "capacidad_kg": q / 100,\n                    "distancia_km": distancia / 1000,\n                    "paradas": paradas,\n                }\n            )\n\n        # El camión vuelve vacío a la base para empezar el siguiente viaje.\n        actual, carga, distancia, paradas = 0, 0, 0.0, []\n\n    # Se recorren los mercados en el orden que indica el cromosoma.\n    for mercado in orden:\n        restante = datos["demandas"][mercado]\n\n        # Mientras quede basura en este mercado, se sigue recogiendo.\n        while restante > 0:\n            # Se recoge lo que quepa: lo que falta del mercado o el espacio libre del camión.\n            recogido = min(restante, q - carga)\n            distancia += matriz[actual][indice[mercado]]  # trayecto hasta el mercado\n            actual = indice[mercado]\n            carga += recogido\n            restante -= recogido\n\n            if detalle:\n                paradas.append({"id_mercado": mercado, "kg": recogido / 100})\n\n            # Restricción de capacidad: si el camión se llenó, va a descargar y empieza otro viaje.\n            # Así un mercado grande (Moshoqueque, 18 800 kg) se reparte entre varios viajes.\n            if carga == q:\n                cerrar()\n\n    # Si quedó carga en el último camión, también se cierra ese viaje.\n    if carga:\n        cerrar()\n\n    # Durante la evolución solo se necesita la distancia total (más rápido).\n    if not detalle:\n        return total\n\n    # Para mostrar resultados se devuelve el plan completo.\n    # Fitness = 1 / (1 + km): a menos kilómetros, mayor aptitud.\n    return {\n        "orden": list(orden),\n        "distancia_km": total / 1000,\n        "fitness": 1 / (1 + total / 1000),\n        "viajes": viajes,\n        "viajes_totales": numero,\n        "demanda_recogida_kg": sum(datos["demandas"].values()) / 100,\n    }\n\n\n# ============================================================\n# APOYO 3. SELECCIÓN POR TORNEO\n# ============================================================\n\ndef seleccionar_torneo(poblacion, costo, tamano, rng):\n    """Elige al azar varios individuos y devuelve el de menor distancia."""\n    # Con la configuración inicial compiten tres individuos por cada padre.\n    aspirantes = rng.sample(poblacion, tamano)\n    return min(aspirantes, key=costo)\n\n\n# ============================================================\n# APOYO 4. CRUCE ORDENADO (OX)\n# ============================================================\n\n# Combina dos padres sin repetir ni perder mercados.\ndef cruce_ox(padre_a, padre_b, rng):\n    """Conserva un segmento de A y completa circularmente con el orden de B."""\n    # Ejemplo con 6 genes: A=[1,2,3,4,5,6], B=[4,6,2,1,5,3], segmento posiciones 2..3.\n    # El hijo copia [_,_,3,4,_,_] de A y rellena el resto con B en orden: [2,1,3,4,5,6].\n    # A diferencia del cruce de un punto, OX nunca repite ni pierde mercados.\n    n = len(padre_a)\n    if n < 2:\n        return padre_a[:]\n\n    # Se eligen dos posiciones distintas al azar: inicio y fin del segmento.\n    inicio, fin = sorted(rng.sample(range(n), 2))\n    fin += 1\n    hijo = [None] * n\n\n    # 1) Se copia el segmento del padre A en la misma posición.\n    hijo[inicio:fin] = padre_a[inicio:fin]\n    usados = set(padre_a[inicio:fin])\n\n    # 2) Se rellenan los huecos desde el final del segmento, dando la vuelta (circular).\n    posicion = fin % n\n\n    # 3) Los genes se toman del padre B en su orden, saltando los que ya están en el hijo.\n    for gen in padre_b[fin:] + padre_b[:fin]:\n        if gen not in usados:\n            hijo[posicion] = gen\n            posicion = (posicion + 1) % n\n\n    return hijo\n\n\n# ============================================================\n# APOYO 5. MUTACIÓN POR INTERCAMBIO (SWAP)\n# ============================================================\n\n# Intercambia dos mercados al azar para explorar órdenes nuevos.\ndef mutar(orden, rng):\n    """Intercambia dos posiciones sin perder ni duplicar mercados."""\n    # Ejemplo: [A,B,C,D] -> se eligen las posiciones 1 y 3 -> [A,D,C,B].\n    hijo = orden[:]  # se trabaja sobre una copia para no alterar al padre\n\n    if len(hijo) >= 2:\n        a, b = rng.sample(range(len(hijo)), 2)\n        hijo[a], hijo[b] = hijo[b], hijo[a]\n\n    return hijo\n', 'datos/15_mercados_chiclayo_11_columnas.csv': 'rango,id_mercado,mercado,distrito,provincia,puestos_fijos_reportados,residuos_kg_dia_estimados,residuos_t_dia_estimados,latitud_wgs84,longitud_wgs84,direccion\n1,CH-1006,Mercado Moshoqueque,José Leonardo Ortiz,Chiclayo,3647,18800.0,18.8,-6.75777,-79.84265,Av. Kennedy cuadra 4 (referencia de acceso)\n2,CH-997,Mercado Modelo Chiclayo,Chiclayo,Chiclayo,910,4690.98,4.69098,-6.76596,-79.83907,Calle Arica s/n\n3,CH-991,Mercado de Abastos José Balta,Chiclayo,Chiclayo,524,2701.18,2.70118,-6.76398,-79.83218,Av. Nicolás de Piérola s/n\n4,CH-990,Mercado Central de Chiclayo,Chiclayo,Chiclayo,512,2639.32,2.63932,-6.77007,-79.83894,Av. José Balta 961\n5,CH-1010,Mercado El Inca,La Victoria,Chiclayo,500,2577.46,2.57746,-6.79837,-79.84471,"Pasaje Las Ñustas s/n, esquina con calle Virú"\n6,CH-1009,Mercado AZ,La Victoria,Chiclayo,500,2577.46,2.57746,-6.80165,-79.83827,"Av. Gran Chimú s/n, frente a vía de evitamiento"\n7,CH-1007,Mercado Pedro Pablo Atusparias,José Leonardo Ortiz,Chiclayo,278,1433.07,1.43307,-6.7595,-79.82534,"Av. Agricultura s/n, altura de la cuadra 3 de calle 5 de Abril"\n8,CH-985,Mercado 9 de Octubre,Chiclayo,Chiclayo,249,1283.58,1.28358,-6.77537,-79.86323,Av. El Ejército s/n\n9,CH-1003,Mercado Carolina,José Leonardo Ortiz,Chiclayo,220,1134.08,1.13408,-6.75065,-79.83144,"Esquina con calle Roma, 4.º sector de Urrunaga"\n10,CH-999,Mercado Sol de Oro,Chiclayo,Chiclayo,200,1030.98,1.03098,-6.78328,-79.86459,Av. Alva Díaz s/n\n11,CH-1011,Mercado Orrego,La Victoria,Chiclayo,143,737.15,0.73715,-6.79262,-79.83456,"Av. Unión 375, esquina con Av. Antenor Orrego"\n12,CH-998,Mercado Santa Rosa de Lima (José Olaya),Chiclayo,Chiclayo,117,603.13,0.60313,-6.7713,-79.85367,Calle Piura s/n\n13,CH-994,Mercado José Abelardo Quiñones Gonzales,Chiclayo,Chiclayo,117,603.13,0.60313,-6.77059,-79.86665,Calle Sargento Lores s/n\n14,CH-984,Mercadito Buenos Aires,Chiclayo,Chiclayo,114,587.66,0.58766,-6.77936,-79.84146,Av. Garcilazo de la Vega 299\n15,CH-987,Mercado Benjamín Gamarra,Chiclayo,Chiclayo,98,505.18,0.50518,-6.7665,-79.83064,Calle Tarapacá 566\n', 'datos/matriz_vial.json': '{\n  "fecha_consulta": "2026-10-15",\n  "fuente": "OSRM Table driving",\n  "unidad": "metros",\n  "orden_ids": [\n    "BASE_EPSEL_CANDIDATO",\n    "CH-1006",\n    "CH-997",\n    "CH-991",\n    "CH-990",\n    "CH-1010",\n    "CH-1009",\n    "CH-1007",\n    "CH-985",\n    "CH-1003",\n    "CH-999",\n    "CH-1011",\n    "CH-998",\n    "CH-994",\n    "CH-984",\n    "CH-987",\n    "BOTADERO_REQUE_CANDIDATO"\n  ],\n  "puntos": [\n    {\n      "id": "BASE_EPSEL_CANDIDATO",\n      "nombre": "Terminal EPSEL",\n      "latitud": -6.7632989,\n      "longitud": -79.8313214,\n      "fuente_gps": "https://www.openstreetmap.org/node/4335559289",\n      "observacion": "Es un grifo cartografiado como Terminal EPSEL. Hay evidencia municipal del depósito en el terminal, pero esta coordenada NO es la puerta verificada del garaje.",\n      "ajuste_a_via_m": 4.423274891\n    },\n    {\n      "id": "CH-1006",\n      "nombre": "Mercado Moshoqueque",\n      "latitud": -6.75777,\n      "longitud": -79.84265,\n      "fuente_gps": "https://mapcarta.com/es/W446750261",\n      "observacion": "Acceso de camión no validado. Masa del escenario representa un ámbito mayor; ubicación es primer sector.",\n      "ajuste_a_via_m": 62.48731543\n    },\n    {\n      "id": "CH-997",\n      "nombre": "Mercado Modelo Chiclayo",\n      "latitud": -6.76596,\n      "longitud": -79.83907,\n      "fuente_gps": "https://mapcarta.com/es/W201959200",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 55.58162971\n    },\n    {\n      "id": "CH-991",\n      "nombre": "Mercado de Abastos José Balta",\n      "latitud": -6.76398,\n      "longitud": -79.83218,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 36.4217883\n    },\n    {\n      "id": "CH-990",\n      "nombre": "Mercado Central de Chiclayo",\n      "latitud": -6.77007,\n      "longitud": -79.83894,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 51.77594221\n    },\n    {\n      "id": "CH-1010",\n      "nombre": "Mercado El Inca",\n      "latitud": -6.79837,\n      "longitud": -79.84471,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 53.68478439\n    },\n    {\n      "id": "CH-1009",\n      "nombre": "Mercado AZ",\n      "latitud": -6.80165,\n      "longitud": -79.83827,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 46.44421485\n    },\n    {\n      "id": "CH-1007",\n      "nombre": "Mercado Pedro Pablo Atusparias",\n      "latitud": -6.7595,\n      "longitud": -79.82534,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 27.74998769\n    },\n    {\n      "id": "CH-985",\n      "nombre": "Mercado 9 de Octubre",\n      "latitud": -6.77537,\n      "longitud": -79.86323,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 18.83310744\n    },\n    {\n      "id": "CH-1003",\n      "nombre": "Mercado Carolina",\n      "latitud": -6.75065,\n      "longitud": -79.83144,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 24.78292232\n    },\n    {\n      "id": "CH-999",\n      "nombre": "Mercado Sol de Oro",\n      "latitud": -6.78328,\n      "longitud": -79.86459,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 18.12969423\n    },\n    {\n      "id": "CH-1011",\n      "nombre": "Mercado Orrego",\n      "latitud": -6.79262,\n      "longitud": -79.83456,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 22.01028227\n    },\n    {\n      "id": "CH-998",\n      "nombre": "Mercado Santa Rosa de Lima (José Olaya)",\n      "latitud": -6.7713,\n      "longitud": -79.85367,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 24.57470377\n    },\n    {\n      "id": "CH-994",\n      "nombre": "Mercado José Abelardo Quiñones Gonzales",\n      "latitud": -6.77059,\n      "longitud": -79.86665,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 25.50267984\n    },\n    {\n      "id": "CH-984",\n      "nombre": "Mercadito Buenos Aires",\n      "latitud": -6.77936,\n      "longitud": -79.84146,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 13.3982339\n    },\n    {\n      "id": "CH-987",\n      "nombre": "Mercado Benjamín Gamarra",\n      "latitud": -6.7665,\n      "longitud": -79.83064,\n      "fuente_gps": "CSV compañero; OSM/Mapcarta declarados, sin URL individual",\n      "observacion": "Acceso de camión no validado.",\n      "ajuste_a_via_m": 32.19225842\n    },\n    {\n      "id": "BOTADERO_REQUE_CANDIDATO",\n      "nombre": "Botadero de Reque",\n      "latitud": -6.9145809,\n      "longitud": -79.7344996,\n      "fuente_gps": "https://www.openstreetmap.org/way/1046386775",\n      "observacion": "Centroide del botadero antiguo solicitado por William; no es acceso validado ni acredita disponibilidad actual. No sustituye al relleno inaugurado en julio 2026.",\n      "ajuste_a_via_m": 279.2231077\n    }\n  ],\n  "distancias_m": [\n    [\n      0,\n      1996.7,\n      1867.2,\n      726.6,\n      1566.8,\n      5524,\n      5602.6,\n      923.1,\n      5264.2,\n      2020.4,\n      5545.7,\n      3949.1,\n      3558.4,\n      5103.4,\n      2753.1,\n      396.4,\n      25252.8\n    ],\n    [\n      2209.9,\n      0,\n      1551.7,\n      1878.2,\n      2088.1,\n      5460.9,\n      5539.5,\n      2383.1,\n      4918,\n      2055,\n      5135,\n      4713.6,\n      2747.7,\n      4075.1,\n      2810.9,\n      2319.9,\n      26736.2\n    ],\n    [\n      1398.6,\n      2010.9,\n      0,\n      1314.8,\n      688.8,\n      4559.5,\n      4638.1,\n      2035.3,\n      4386.2,\n      2631.4,\n      4233.6,\n      3812.2,\n      2680.4,\n      4225.4,\n      2126.8,\n      1173.7,\n      25129.8\n    ],\n    [\n      331.7,\n      2122.3,\n      1992.7,\n      0,\n      1432.1,\n      5569.3,\n      5647.8,\n      968.4,\n      5129.5,\n      2145.9,\n      5411,\n      3994.3,\n      3423.6,\n      4968.6,\n      2771.3,\n      441.7,\n      25298\n    ],\n    [\n      1597.4,\n      1989.7,\n      828.3,\n      1567,\n      0,\n      4213.7,\n      4292.3,\n      2234.1,\n      3694.8,\n      2845.8,\n      3887.9,\n      3466.4,\n      1988.9,\n      3533.9,\n      1563.7,\n      1372.4,\n      24844.8\n    ],\n    [\n      5805.5,\n      5514.6,\n      4516.5,\n      5641.3,\n      4755.5,\n      0,\n      1236.6,\n      5981.7,\n      4011.1,\n      6927.1,\n      4001.4,\n      1691,\n      3798.4,\n      5197.9,\n      2991,\n      5037.6,\n      22433.3\n    ],\n    [\n      5856.5,\n      5565.6,\n      4567.5,\n      5692.3,\n      4806.5,\n      1249.7,\n      0,\n      6032.7,\n      5063.7,\n      6978.1,\n      4998.3,\n      1736.7,\n      4851,\n      6250.6,\n      3042,\n      5088.6,\n      21251.2\n    ],\n    [\n      1017.9,\n      2808.4,\n      2678.9,\n      1538.3,\n      2378.5,\n      5991.4,\n      6069.9,\n      0,\n      6075.9,\n      1548.1,\n      6220.7,\n      4416.4,\n      4370.1,\n      5915.1,\n      3535.5,\n      1197.3,\n      25720.1\n    ],\n    [\n      4947,\n      4396.5,\n      3784.9,\n      4909.6,\n      4023.9,\n      4174.6,\n      5191.7,\n      5583.7,\n      0,\n      6195.4,\n      1274.5,\n      4841.7,\n      1784.1,\n      1186.9,\n      3356.2,\n      4722.1,\n      26388.4\n    ],\n    [\n      2277.1,\n      2048.5,\n      3086,\n      1945.4,\n      2785.7,\n      7026,\n      7104.6,\n      1536.1,\n      6483,\n      0,\n      6764.6,\n      6451.6,\n      4777.2,\n      6322.2,\n      3952.1,\n      2387.1,\n      27755.3\n    ],\n    [\n      5377.2,\n      4915.7,\n      4215,\n      5339.8,\n      4109,\n      4366,\n      4822.7,\n      6307.2,\n      1295,\n      6625.6,\n      0,\n      4472.6,\n      2544.7,\n      2481.9,\n      2987.2,\n      5040.6,\n      26019.4\n    ],\n    [\n      4234.4,\n      4691.2,\n      3693.1,\n      4187.7,\n      3143.6,\n      1710.6,\n      1783,\n      4410.6,\n      4711.8,\n      6048.6,\n      4646.5,\n      0,\n      4065.8,\n      5898.7,\n      2167.6,\n      3466.5,\n      21917.5\n    ],\n    [\n      3417.3,\n      2667.3,\n      2255.1,\n      3379.9,\n      2494.1,\n      3896.5,\n      5098.3,\n      4054,\n      1680.3,\n      4665.7,\n      2434.6,\n      4272.4,\n      0,\n      2008.5,\n      2369.6,\n      3192.3,\n      26295\n    ],\n    [\n      4958.7,\n      4168.6,\n      3796.5,\n      4921.3,\n      4035.5,\n      6495.6,\n      6639.7,\n      5595.4,\n      1086.8,\n      6207.1,\n      1695,\n      5813.8,\n      1795.8,\n      0,\n      3911.1,\n      4733.8,\n      27836.4\n    ],\n    [\n      3047,\n      2882.9,\n      1884.9,\n      3009.6,\n      2123.9,\n      2893.1,\n      2971.7,\n      3977.1,\n      3336.8,\n      4295.4,\n      3164.6,\n      2145.8,\n      2257.6,\n      4100,\n      0,\n      2710.5,\n      24168.4\n    ],\n    [\n      615.8,\n      2406.4,\n      1736.1,\n      682.1,\n      1300.2,\n      5121.9,\n      5200.5,\n      1206.1,\n      4908.5,\n      2430.1,\n      5351.2,\n      3547,\n      3202.7,\n      4747.7,\n      2666.1,\n      0,\n      24850.7\n    ],\n    [\n      25393.8,\n      26606.2,\n      25608.1,\n      25347.1,\n      24303,\n      22290.3,\n      21212.2,\n      25570,\n      26104.3,\n      27208,\n      26038.9,\n      21724.2,\n      25891.6,\n      27291.1,\n      24082.6,\n      24625.9,\n      0\n    ]\n  ],\n  "nota": "Submatriz de respuesta OSRM de 47 puntos. Perfil automovil, rutas rapidas; sin trafico en vivo ni restricciones de camion. Extremos referenciales.",\n  "consulta_original": "https://router.project-osrm.org/table/v1/driving/-79.84265,-6.75777;-79.83907,-6.76596;-79.83218,-6.76398;-79.83894,-6.77007;-79.83827,-6.80165;-79.84471,-6.79837;-79.872193,-6.878392;-79.82534,-6.7595;-79.862684,-6.907086;-79.559323,-6.889762;-79.86323,-6.77537;-79.83144,-6.75065;-79.30306,-6.847197;-79.637679,-6.736646;-79.86459,-6.78328;-79.38691,-6.642387;-79.609231,-6.781147;-79.9354055,-6.8347728;-79.83456,-6.79262;-79.77448,-6.76865;-79.86665,-6.77059;-79.85367,-6.7713;-79.84146,-6.77936;-79.81845,-6.866474;-79.83064,-6.7665;-79.922955,-6.880533;-79.702017,-6.748314;-79.8301283,-6.754552;-79.8424,-6.78748;-79.562753,-6.891039;-79.85453,-6.76422;-79.83203,-6.78332;-79.582724,-6.923274;-79.62127,-6.991856;-79.84631,-6.78047;-79.82017,-6.75258;-79.83246,-6.7758;-79.8461143,-6.7759955;-79.869395,-6.874712;-79.84603,-6.78049;-79.83125,-6.76056;-79.63829,-6.81438;-79.602875,-6.810391;-79.83432,-6.7724;-79.8313214,-6.7632989;-79.7344996,-6.9145809;-79.7404168,-6.8839258?annotations=distance,duration",\n  "version_red_vial": "No declarada por el servidor"\n}', 'datos/configuracion.json': '{\n  "capacidad_kg": 10000,\n  "camiones": 2,\n  "viajes_por_camion": 3,\n  "poblacion": 100,\n  "generaciones": 500,\n  "prob_cruce": 0.9,\n  "prob_mutacion": 0.2,\n  "elite": 2,\n  "torneo": 3,\n  "paciencia": 100,\n  "semilla": 42\n}'}
for nombre, contenido in archivos.items():
    destino = RAIZ_NOTEBOOK / nombre
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(contenido, encoding="utf-8")
sys.path.insert(0, str(RAIZ_NOTEBOOK / 'con_comentarios'))


In [ ]:
from sistema.lector_datos import cargar_datos
from algoritmo_genetico.motor_genetico import optimizar
configuracion = json.loads((RAIZ_NOTEBOOK / "datos/configuracion.json").read_text(encoding="utf-8"))
resultado = optimizar(cargar_datos(), configuracion)
print("Referencia km:", resultado["referencia"]["distancia_km"])
print("AG km:", resultado["solucion"]["distancia_km"])
print("Mejora %:", resultado["mejora_porcentaje"])
print("Generaciones:", resultado["generaciones_ejecutadas"])
for viaje in resultado["solucion"]["viajes"]:
    print(viaje)
(RAIZ_NOTEBOOK / "resultado_rutas.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")
